# OceanEmbed — end-to-end Colab run (SIH26066)

**Satellite-based reconstruction of subsurface ocean temperature (0–1000 m)** from
surface-only satellite observations, using **100% real data** (Argo floats + NOAA OISST,
optionally CMEMS SSH and SMAP SSS).

This notebook runs the whole pipeline top-to-bottom on a Google Colab runtime:
**ingestion → collocation → training → evaluation**.

### Before you run

1. Upload the entire `oceanembed/` repository folder into Colab's file browser
   (or `git clone` it inside the runtime), then open this notebook **inside that folder**.
2. (Optional) For SSH/SSS features: copy `.env.example` → `.env` and add your free
   CMEMS / NASA Earthdata credentials. Without them the pipeline runs in *degraded mode*
   (SST only) — everything still uses real data.
3. Runtime → *Change runtime type* → **T4 GPU** (optional but faster for the MLP).

Everything below the line executes the actual pipeline CLI entry points; no cells
invent data.


In [ ]:
#@title 1. Environment setup (run once)
import os, sys, subprocess
from pathlib import Path

REPO = Path.cwd()
# NOTE: if you uploaded the repo as a folder, cwd is likely:
# /content/oceanembed. If needed, fix REPO manually:
# REPO = Path('/content/oceanembed')

os.chdir(REPO)

# 1. system deps needed by some wheels
subprocess.run(['apt-get', 'install', '-y', 'libgomp1', 'libgl1'], check=False,
               stdout=subprocess.DEVNULL)

# 2. install project deps (tensorflow is preinstalled on Colab; pip will keep it)
%pip install -q -r requirements.txt
%pip install -q python-dotenv

# 3. load optional credentials from .env (if the user uploaded one)
from dotenv import load_dotenv
load_dotenv(REPO / '.env')
print('env ready; GPU:', end=' ')
import tensorflow as tf
print(tf.config.list_physical_devices('GPU') or 'CPU only')


## Phase 1 — Ingestion of real data

* `fetch_argo`  → Argo float T(z) profiles (argopy GDAC / Argovis fallback), no login.
* `fetch_oisst` → NOAA OISST daily SST (public ERDDAP), no login.
* `fetch_copernicus` / `fetch_smap` → skip gracefully when `.env` credentials are missing.

The quick-start config (`config.bootstrap.yaml`) uses a ~13-month window so Colab free
tiers and laptops finish quickly. Use `config.yaml` (5-year window) for the full run.


In [ ]:
#@title 2. Ingestion (real data only)
CONFIG = "config.bootstrap.yaml"   #@param ["config.bootstrap.yaml", "config.yaml"]
!python -m src.ingestion.fetch_all --config {CONFIG}


## Phase 2 — Collocation

Argo profiles are matched to the nearest satellite grid cell (KDTree, ±2-day window),
interpolated onto the 6 standard depth levels (0/50/100/200/500/1000 m), and written to
`data/interim/collocated.parquet` with a matching report.


In [ ]:
#@title 3. Collocation
!python -m src.collocation.collocate --config {CONFIG}


## Phase 3–4 — Baseline + deep models

`train.py` splits **by Argo float ID** (no leakage), trains Random Forest, XGBoost and a
Keras MLP (multi-output — all six depths in one pass), and saves artifacts to `models/`.


In [ ]:
#@title 4. Train (baselines + deep MLP)
!python -m src.models.train --config {CONFIG}


## Phase 5 — Evaluation

Per-depth RMSE / MAE / R² table, predicted-vs-actual profile plots, and a
plain-language summary.


In [ ]:
#@title 5. Evaluate
!python -m src.evaluation.evaluate --config {CONFIG}
print()
print(open('reports/results_summary.md').read())


## Results figures

Inspect the comparison table and the sample predicted-vs-actual profiles.


In [ ]:
#@title 6. Show comparison table + profile figures
import pandas as pd, glob
from IPython.display import display, Image

table = pd.read_csv('reports/metrics_comparison.csv')
display(table)

for f in sorted(glob.glob('reports/figures/profile_*.png'))[:3]:
    display(Image(f, width=420))


## Optional — interactive demo

After training, run the offline Streamlit map on your Colab machine:

```
!streamlit run app/streamlit_app.py --server.port 8501 &
```

or locally with `streamlit run app/streamlit_app.py`.

---

### What was skipped?

Check the raw-source manifests for status: `downloads ok` vs `skipped (missing
credentials)`. CNN-LSTM patch model and full 5-year/global window are next-milestone
items (see README).
